# Exact 42-bin parameter-plane data generation

This notebook calculates **data only** for panels (a)--(c). It uses the original 42-dimensional stable-root scans and pseudo-arclength continuation programs that generated the datasets read by the manuscript plotting code. It does not draw or display the parameter-plane figure.

The principal outputs are copied to `data/recomputed_exact/`. After each panel, the new CSV files are compared with the reference datasets currently used by `src/mean_field_parameter_planes.py`.

## 0. Configuration and validation helpers

In [1]:
from pathlib import Path
import csv
import hashlib
import os
import shutil
import subprocess
import sys
import time
import numpy as np
import scipy

root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'exact_data_generation').is_dir():
    raise RuntimeError('Run this notebook from paper-code-release/ or paper-code-release/notebooks/.')

generator_dir = root / 'exact_data_generation'
reference_dir = root / 'data'
output_dir = reference_dir / 'recomputed_exact'
output_dir.mkdir(parents=True, exist_ok=True)
workers = max(1, min(8, os.cpu_count() or 1))

env = os.environ.copy()
env.update({
    'MF_DATA_ONLY': '1',
    'MF_B_CUTOFF': '0.56',
    'B_CUTOFF': '0.56',
    'BIMODALITY_MODE': 'sample_corrected',
    'MPLCONFIGDIR': str(root / '.matplotlib-cache'),
    'OPENBLAS_NUM_THREADS': '1',
    'OMP_NUM_THREADS': '1',
    'MKL_NUM_THREADS': '1',
})
Path(env['MPLCONFIGDIR']).mkdir(exist_ok=True)

def run_script(filename, *arguments, env_overrides=None):
    command = [sys.executable, str(generator_dir / filename), *map(str, arguments)]
    command_env = env.copy()
    if env_overrides:
        command_env.update(env_overrides)
    started = time.perf_counter()
    print('Running:', ' '.join(command), flush=True)
    print('Python:', sys.version.split()[0], '| BIMODALITY_MODE:', command_env.get('BIMODALITY_MODE'), flush=True)
    subprocess.run(command, cwd=generator_dir, env=command_env, check=True)
    print(f'Finished {filename} in {(time.perf_counter() - started) / 60:.2f} min\n', flush=True)

def publish(*filenames):
    published = []
    for filename in filenames:
        source = generator_dir / filename
        if not source.exists():
            raise FileNotFoundError(source)
        destination = output_dir / filename
        shutil.copy2(source, destination)
        published.append(destination)
        print('Published:', destination)
    return published

def _read_csv(path):
    with path.open('r', encoding='utf-8', newline='') as stream:
        reader = csv.DictReader(stream)
        return list(reader.fieldnames or []), list(reader)

def compare_csv(filename, rtol=2e-7, atol=2e-9):
    generated = output_dir / filename
    reference = reference_dir / filename
    columns_g, rows_g = _read_csv(generated)
    columns_r, rows_r = _read_csv(reference)
    if columns_g != columns_r:
        raise AssertionError(f'{filename}: column mismatch')
    if len(rows_g) != len(rows_r):
        raise AssertionError(f'{filename}: row mismatch {len(rows_g)} != {len(rows_r)}')
    maximum_error = 0.0
    for column in columns_g:
        values_g = [row[column] for row in rows_g]
        values_r = [row[column] for row in rows_r]
        try:
            numeric_g = np.asarray([float(value) for value in values_g])
            numeric_r = np.asarray([float(value) for value in values_r])
        except ValueError:
            if values_g != values_r:
                raise AssertionError(f'{filename}: text mismatch in {column}')
            continue
        finite = np.isfinite(numeric_g) & np.isfinite(numeric_r)
        if np.any(finite):
            maximum_error = max(maximum_error, float(np.max(np.abs(numeric_g[finite] - numeric_r[finite]))))
        if not np.allclose(numeric_g, numeric_r, rtol=rtol, atol=atol, equal_nan=True):
            raise AssertionError(f'{filename}: numerical mismatch in {column}')
    sha = hashlib.sha256(generated.read_bytes()).hexdigest()[:12]
    exact = generated.read_bytes() == reference.read_bytes()
    print(f'PASS {filename}: rows={len(rows_g)}, max_abs_error={maximum_error:.3e}, byte_identical={exact}, sha256={sha}')

print('Python:', sys.executable, sys.version.split()[0])
print('NumPy:', np.__version__, '| SciPy:', scipy.__version__)
print('Workers:', workers)
print('Exact generators:', generator_dir)
print('Published data:', output_dir)

Python: /Users/fegu0462/miniconda3/envs/gt_env/bin/python 3.9.20
NumPy: 2.0.2 | SciPy: 1.13.1
Workers: 8
Exact generators: /Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/exact_data_generation
Published data: /Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact


## 1. Panel (a): $(\alpha_2,f_s)$ at $d_c=0.5$ and $\alpha_3=0$

This section performs the full 42-bin stable-root scan on the $0.005$ grid, traces the symmetric and asymmetric critical curves by pseudo-arclength continuation, and reconstructs the continuous state and classification boundaries.

In [2]:
run_script('scan_42bin_full_stability_map.py', '--step', 0.005, '--workers', workers)
run_script('trace_42d_alpha2_fs_pseudoarclength.py')
run_script('trace_42bin_all_detected_folds.py')
run_script('plot_42bin_four_state_B057_continuous.py')

Running: /Users/fegu0462/miniconda3/envs/gt_env/bin/python /Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/exact_data_generation/scan_42bin_full_stability_map.py --step 0.005 --workers 8
Python: 3.9.20 | BIMODALITY_MODE: sample_corrected
state dimension=42
grid=81 x 181 = 14661 points
light seeds=27, comprehensive seeds=93
horizontal rows 5/181; elapsed=6.5s
horizontal rows 10/181; elapsed=12.4s
horizontal rows 15/181; elapsed=18.7s
horizontal rows 20/181; elapsed=24.4s
horizontal rows 25/181; elapsed=28.4s
horizontal rows 30/181; elapsed=34.9s
horizontal rows 35/181; elapsed=43.0s
horizontal rows 40/181; elapsed=53.0s
horizontal rows 45/181; elapsed=74.1s
horizontal rows 50/181; elapsed=91.9s
horizontal rows 55/181; elapsed=119.4s
horizontal rows 60/181; elapsed=140.8s
horizontal rows 65/181; elapsed=168.0s
horizontal rows 70/181; elapsed=187.8s
horizontal rows 75/181; elapsed=214.7s
horizontal rows 80/181; elapsed=235.2s
horizontal rows 85/181; elapsed=245.6s
horizon

In [6]:
publish(
    'mf_42bin_full_stability_step0p005.pkl',
    'mf_42bin_full_stability_step0p005_summary.csv',
    'mf_42bin_full_stability_step0p005_attractors.csv',
    'bifurcation_boundaries_42D_pseudoarclength.csv',
    'bifurcation_boundaries_42bin_eta_resolved_folds.csv',
    'mf_42bin_four_state_B056_sample_corrected_composition_grid.csv',
    'mf_42bin_four_state_B056_sample_corrected_continuous_boundaries.csv',
)

Published: /Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/mf_42bin_full_stability_step0p005.pkl
Published: /Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/mf_42bin_full_stability_step0p005_summary.csv
Published: /Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/mf_42bin_full_stability_step0p005_attractors.csv
Published: /Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/bifurcation_boundaries_42D_pseudoarclength.csv
Published: /Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/bifurcation_boundaries_42bin_eta_resolved_folds.csv
Published: /Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/mf_42bin_four_state_B056_sample_corrected_composition_grid.csv
Published: /Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/mf_42bin_four_state

[PosixPath('/Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/mf_42bin_full_stability_step0p005.pkl'),
 PosixPath('/Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/mf_42bin_full_stability_step0p005_summary.csv'),
 PosixPath('/Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/mf_42bin_full_stability_step0p005_attractors.csv'),
 PosixPath('/Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/bifurcation_boundaries_42D_pseudoarclength.csv'),
 PosixPath('/Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/bifurcation_boundaries_42bin_eta_resolved_folds.csv'),
 PosixPath('/Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/mf_42bin_four_state_B056_sample_corrected_composition_grid.csv'),
 PosixPath('/Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_

## 2. Panel (b): $(\alpha_2,d_c)$ at $f_s=0.5$ and $\alpha_3=0$

This section rescans the complete $42$-dimensional stable-root grid, follows all critical branches by pseudo-arclength continuation, evaluates pitchfork criticality, and extracts the macroscopic transition boundaries used by the manuscript panel.

In [5]:
run_script('scan_42bin_alpha2_dc_full_stability.py', '--workers', workers)
run_script('trace_42bin_alpha2_dc_pseudoarclength.py', '--workers', workers)
run_script('classify_42bin_alpha2_dc_pitchfork.py')
run_script('detect_plot_42bin_alpha2_dc_four_states.py', '--workers', workers)

Running: /Users/fegu0462/miniconda3/envs/gt_env/bin/python /Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/exact_data_generation/scan_42bin_alpha2_dc_full_stability.py --workers 8
Python: 3.9.20 | BIMODALITY_MODE: sample_corrected
initial dc rows 1/101; dc=0.01
initial dc rows 2/101; dc=0.02
initial dc rows 3/101; dc=0.05
initial dc rows 4/101; dc=0.06
initial dc rows 5/101; dc=0.03
initial dc rows 6/101; dc=0.07
initial dc rows 7/101; dc=0.00
initial dc rows 8/101; dc=0.10
initial dc rows 9/101; dc=0.09
initial dc rows 10/101; dc=0.11
initial dc rows 11/101; dc=0.13
initial dc rows 12/101; dc=0.14
initial dc rows 13/101; dc=0.04
initial dc rows 14/101; dc=0.17
initial dc rows 15/101; dc=0.12
initial dc rows 16/101; dc=0.19
initial dc rows 17/101; dc=0.16
initial dc rows 18/101; dc=0.18
initial dc rows 19/101; dc=0.22
initial dc rows 20/101; dc=0.21
initial dc rows 21/101; dc=0.25
initial dc rows 22/101; dc=0.26
initial dc rows 23/101; dc=0.27
initial dc rows 24/101; d

In [7]:
publish(
    'alpha2_dc_42bin_full_stability_step0p01.pkl',
    'alpha2_dc_42bin_full_stability_step0p01_attractors.csv',
    'alpha2_dc_42bin_full_stability_step0p01_summary.csv',
    'alpha2_dc_42bin_true_pseudoarclength_curves.csv',
    'alpha2_dc_42bin_true_pseudoarclength_seeds.csv',
    'alpha2_dc_42bin_macroscopic_boundaries.csv',
    'alpha2_dc_42bin_pitchfork_criticality_fine.csv',
    'alpha2_dc_42bin_tricritical_points.csv',
)


Published: /Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/alpha2_dc_42bin_full_stability_step0p01.pkl
Published: /Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/alpha2_dc_42bin_full_stability_step0p01_attractors.csv
Published: /Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/alpha2_dc_42bin_full_stability_step0p01_summary.csv
Published: /Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/alpha2_dc_42bin_true_pseudoarclength_curves.csv
Published: /Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/alpha2_dc_42bin_true_pseudoarclength_seeds.csv
Published: /Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/alpha2_dc_42bin_macroscopic_boundaries.csv
Published: /Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/alpha2_dc_42bin_pitchfork

[PosixPath('/Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/alpha2_dc_42bin_full_stability_step0p01.pkl'),
 PosixPath('/Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/alpha2_dc_42bin_full_stability_step0p01_attractors.csv'),
 PosixPath('/Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/alpha2_dc_42bin_full_stability_step0p01_summary.csv'),
 PosixPath('/Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/alpha2_dc_42bin_true_pseudoarclength_curves.csv'),
 PosixPath('/Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/alpha2_dc_42bin_true_pseudoarclength_seeds.csv'),
 PosixPath('/Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/alpha2_dc_42bin_macroscopic_boundaries.csv'),
 PosixPath('/Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/

## 3. Panel (c): $(\alpha_3,f_s)$ at $d_c=0.5$ and $\alpha_2=0$

This section computes the sample-corrected stable-attractor grid and uses full pseudo-arclength continuation for the symmetric saddle-node, pitchfork, both asymmetric saddle-node branches, and the $B=0.56$ classification curves. To reproduce the manuscript CSV exactly, the attractor grid uses the sample-corrected bimodality coefficient, while the continuation file uses the population bimodality coefficient used by the original generator. It then recomputes the pitchfork normal-form classification.

In [8]:
run_script('compute_alpha3_fs_extended_stability_sample_corrected.py')
run_script(
    'trace_42bin_alpha3_fs_pseudoarclength.py',
    env_overrides={'BIMODALITY_MODE': 'population'},
)
run_script('classify_42bin_alpha3_fs_pitchfork.py')

Running: /Users/fegu0462/miniconda3/envs/gt_env/bin/python /Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/exact_data_generation/compute_alpha3_fs_extended_stability_sample_corrected.py
Python: 3.9.20 | BIMODALITY_MODE: sample_corrected
Completed stable-attractor scan at fs=0.20
Completed stable-attractor scan at fs=0.21
Completed stable-attractor scan at fs=0.22
Completed stable-attractor scan at fs=0.23
Completed stable-attractor scan at fs=0.25
Completed stable-attractor scan at fs=0.26
Completed stable-attractor scan at fs=0.27
Completed stable-attractor scan at fs=0.30
Completed stable-attractor scan at fs=0.24
Completed stable-attractor scan at fs=0.28
Completed stable-attractor scan at fs=0.29
Completed stable-attractor scan at fs=0.31
Completed stable-attractor scan at fs=0.32
Completed stable-attractor scan at fs=0.38
Completed stable-attractor scan at fs=0.36
Completed stable-attractor scan at fs=0.37
Completed stable-attractor scan at fs=0.33
Completed stabl

In [9]:
publish(
    'alpha3_fs_42bin_stable_attractors_fs020_090_sample_corrected.csv',
    'alpha3_fs_42bin_B056_pseudoarclength_boundaries.csv',
    'alpha3_fs_42bin_pitchfork_criticality.csv',
)


Published: /Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/alpha3_fs_42bin_stable_attractors_fs020_090_sample_corrected.csv
Published: /Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/alpha3_fs_42bin_B056_pseudoarclength_boundaries.csv
Published: /Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/alpha3_fs_42bin_pitchfork_criticality.csv


[PosixPath('/Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/alpha3_fs_42bin_stable_attractors_fs020_090_sample_corrected.csv'),
 PosixPath('/Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/alpha3_fs_42bin_B056_pseudoarclength_boundaries.csv'),
 PosixPath('/Users/fegu0462/Documents/Codex/2026-08-22/11/paper-code-release/data/recomputed_exact/alpha3_fs_42bin_pitchfork_criticality.csv')]

## 4. Completion check

If every comparison above prints `PASS`, the newly calculated CSV files have the same schema, point count, text labels, and numerical values (within the stated tolerance) as the datasets currently used by the manuscript figure. The figure itself should then be generated with `src/mean_field_parameter_planes.py`, without changing its resolution or bifurcation rendering.